# X-ray mask generation interactive notebook

This notebook is the recommended interactive front end for `pattern_generation.py`. It generates binary mask arrays for simulation and exports PNG, TIFF, NPY, JSON metadata, compact GDSII layouts, and central PNG inspection figures. All geometric inputs are physical units, not pixels.

## Parameter reference

Shared parameters:

- `field_size_um`: `(height_um, width_um)` total mask size. Example: `(1000, 1000)` is 1 mm by 1 mm.
- `raster_resolution_um`: output pixel size in micrometers. Smaller values improve simulation fidelity but increase memory. For hard X-ray masks, `0.25` um is usually practical; for soft X-ray fine patterns, use `0.05` um or finer if memory allows.
- `xray_energy_keV`: photon energy used for wavelength metadata and absorber estimates.
- `polarity`: `"holes_transmit"` means generated features are transparent pixels (`1`). `"absorber_transmit"` inverts the mask.
- `supersample`: oversampling factor for rasterization. Use `4` for circular or rotated features. Use `1` for exact square random-cell masks.

Export methods:

- `to_array()` returns a NumPy `uint8` array where `1` is transmitting.
- `save_all(output_dir, stem)` writes `{stem}.png`, `{stem}.tiff`, `{stem}.npy`, `{stem}.json`, and `{stem}_preview.png`.
- `save_fabrication_outputs(pattern, stem, arr)` writes `{stem}.gds` using the current compact tiled GDSII format and `{stem}_center.png`, a nearest-neighbor enlarged center crop.
- `save_metadata(path)` writes exact parameters and derived wavelength. For `RandomHartmann`, it also stores every hole center, size, and rotation. For `DammannGrating`, it records optimized transitions, target-order uniformity, and focal-plane beam spacing.

In [1]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_ROOT = Path.cwd().resolve()
if not (NOTEBOOK_ROOT / "scripts").is_dir():
    NOTEBOOK_ROOT = NOTEBOOK_ROOT.parent
sys.path.insert(0, str(NOTEBOOK_ROOT / "scripts"))

from export_fabrication_patterns import _save_binary_gds, _save_center_png, save_fixed_depth_gds, save_random_hartmann_gds
from grating_design import DesignConfig
from pattern_comparison import default_patterns, simulate_pattern
from pattern_generation import (
    Checkerboard,
    DammannGrating,
    Grating,
    HartmannPlate,
    KinoformCRL1D,
    RandomBinaryMask,
    RandomHartmann,
    absorber_requirements,
    analytic_hartmann_first_guess,
)

OUTPUT_DIR = Path("generated_patterns")
OUTPUT_DIR.mkdir(exist_ok=True)

def show_mask(pattern, title="mask", max_pixels=1200):
    # RandomHartmann has a sparse bounded preview path. It deliberately does
    # not create the full fabrication raster just to display a 1200 px plot.
    arr = pattern.preview_array(max_pixels) if isinstance(pattern, RandomHartmann) else pattern.to_array()
    stride = max(1, int(np.ceil(max(arr.shape) / max_pixels)))
    plt.figure(figsize=(6, 6))
    plt.imshow(arr[::stride, ::stride], cmap="gray", origin="lower", interpolation="nearest")
    plt.title(title)
    plt.xlabel("x pixels")
    plt.ylabel("y pixels")
    plt.show()
    if isinstance(pattern, RandomHartmann):
        h_um, w_um = pattern.field_size_um
        print("preview shape y,x:", arr.shape, "fabrication shape y,x:", (int(np.ceil(h_um / pattern.raster_resolution_um)), int(np.ceil(w_um / pattern.raster_resolution_um))), "preview transmitting fraction:", arr.mean())
    else:
        print("shape y,x:", arr.shape, "transmitting fraction:", arr.mean())
    return arr

def save_fabrication_outputs(pattern, stem, arr=None, center_crop_px=1024, center_output_px=2048):
    """Write the generated binary mask as compact GDSII and a central PNG crop."""
    arr = pattern.to_array() if arr is None else arr
    mask = np.asarray(arr, dtype=bool)
    pixel_um = float(pattern.raster_resolution_um)
    safe_name = "".join(ch if ch.isalnum() else "_" for ch in stem.upper())[:20] or "MASK"
    gds_path = OUTPUT_DIR / f"{stem}.gds"
    center_path = OUTPUT_DIR / f"{stem}_center.png"
    _save_binary_gds(mask, gds_path, pixel_um, mask.shape[0], mask.shape[1], safe_name)
    _save_center_png(mask, center_path, center_crop_px, center_output_px)
    return {"gds": gds_path, "center_png": center_path}

def save_random_hartmann_outputs(pattern, stem, preview, center_output_px=2048):
    """Export sparse apertures as a fabrication-grid-aligned vector GDS.

    This avoids materializing the 65,000 x 65,000 fabrication raster.
    The GDS vertices are quantized to pattern.raster_resolution_um.
    """
    holes = pattern.holes or pattern.generate_holes()
    safe_name = "".join(ch if ch.isalnum() else "_" for ch in stem.upper())[:20] or "RANDOM_HARTMANN"
    gds_path = OUTPUT_DIR / f"{stem}.gds"
    preview_path = OUTPUT_DIR / f"{stem}_preview.png"
    metadata_path = OUTPUT_DIR / f"{stem}.json"
    save_random_hartmann_gds(holes, gds_path, pattern.field_size_um, pattern.raster_resolution_um, safe_name, pattern.hole_shape, pattern.n_sides)
    _save_center_png(np.asarray(preview, dtype=bool), preview_path, min(preview.shape), center_output_px)
    metadata = pattern.metadata(array_shape=(int(np.ceil(pattern.field_size_um[0] / pattern.raster_resolution_um)), int(np.ceil(pattern.field_size_um[1] / pattern.raster_resolution_um))))
    metadata.update({"export_mode": "sparse_vector_grid_quantized", "gds_coordinate_grid_um": pattern.raster_resolution_um, "preview_raster_shape_yx": list(preview.shape), "full_raster_materialized": False})
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return {"gds": gds_path, "preview_png": preview_path, "metadata": metadata_path}

def save_periodic_dammann_gds(pattern, stem, field_size_um, raster_resolution_um, center_crop_px=1024, center_output_px=2048, min_pixels_per_period=16):
    """Export a large periodic Dammann aperture without materializing its full raster."""
    period_pixels = int(round(pattern.period_um / raster_resolution_um))
    if period_pixels < min_pixels_per_period or not np.isclose(period_pixels * raster_resolution_um, pattern.period_um, rtol=0, atol=1e-9):
        raise ValueError(f"fabrication raster must resolve one Dammann period with an integer number of pixels (at least {min_pixels_per_period})")
    tile = DammannGrating(
        field_size_um=(pattern.period_um, pattern.period_um),
        raster_resolution_um=raster_resolution_um,
        xray_energy_keV=pattern.xray_energy_keV,
        orders_per_axis=pattern.orders_per_axis,
        period_um=pattern.period_um,
        focal_length_m=pattern.focal_length_m,
        transition_count=pattern.transition_count,
        optimization_iterations=0,
        transitions_half_period=pattern.design_transitions(),
        include_binary_fresnel_lens=False,
    ).to_array()
    rows = int(np.ceil(field_size_um[0] / raster_resolution_um))
    cols = int(np.ceil(field_size_um[1] / raster_resolution_um))
    safe_name = "".join(ch if ch.isalnum() else "_" for ch in stem.upper())[:20] or "DAMMANN"
    gds_path = OUTPUT_DIR / f"{stem}.gds"
    center_path = OUTPUT_DIR / f"{stem}_center.png"
    _save_binary_gds(tile, gds_path, raster_resolution_um, rows, cols, safe_name)
    crop_size = min(center_crop_px, max(rows, cols))
    repeats_y = int(np.ceil(crop_size / tile.shape[0])) + 1
    repeats_x = int(np.ceil(crop_size / tile.shape[1])) + 1
    preview = np.tile(tile, (repeats_y, repeats_x))[:crop_size, :crop_size]
    _save_center_png(preview, center_path, crop_size, center_output_px)
    metadata_path = OUTPUT_DIR / f"{stem}.json"
    metadata = pattern.metadata(array_shape=(rows, cols))
    metadata.update({"export_mode": "periodic_gds_aref", "fabrication_field_size_um": list(field_size_um), "fabrication_raster_resolution_um": raster_resolution_um, "period_tile_shape_px": list(tile.shape)})
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return {"gds": gds_path, "center_png": center_path, "metadata": metadata_path}


## 1D grating and 2D grid

Important parameters:

- `period_um`: repeat distance.
- `duty_cycle`: transmitting width divided by period. Must be between `0` and `1`.
- `dimensionality`: `"1D"` for line grating or `"2D"` for crossed grating.
- `orientation_deg`: rotates the pattern in degrees.
- `combine_mode`: for `"2D"`, `"and"` gives isolated transmitting squares; `"or"` gives transmitting bars/posts depending on polarity.
- `period_x_um`, `period_y_um`, `duty_cycle_x`, `duty_cycle_y`: optional independent x/y settings.

In [6]:
grid = Grating(
    field_size_um=(13000, 13000),
    raster_resolution_um=2,
    xray_energy_keV=14.0,
    period_um=10.0,
    duty_cycle=0.5,
    dimensionality="2D",
    combine_mode="and",
    orientation_deg=0.0,
    supersample=1,
)
arr = show_mask(grid, "2D grid, 10 um period")
# grid.save_all(OUTPUT_DIR, "grid_10um_13mm")
save_fabrication_outputs(grid, "grid_10um_13mm", arr)

C:\Users\qiaoz\AppData\Local\Temp\ipykernel_60848\1426181235.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


shape y,x: (6500, 6500) transmitting fraction: 0.36


{'gds': WindowsPath('generated_patterns/grid_10um_13mm.gds'),
 'center_png': WindowsPath('generated_patterns/grid_10um_13mm_center.png')}

## Checkerboard

Important parameters:

- `period_um`: full two-tile repeat. Each standard tile is `period_um / 2`.
- `duty_cycle`: leave at `0.5` for a standard checkerboard. Non-0.5 values gate the checkerboard with a rectangular duty envelope.
- `orientation_deg`: use `45` for a rotated Talbot-style checkerboard.

In [22]:
checker = Checkerboard(
    field_size_um=(13000, 13000),
    raster_resolution_um=2.5,
    xray_energy_keV=14.0,
    period_um=10,
    orientation_deg=0.0,
    supersample=1,
)
arr = show_mask(checker, "0 degree checkerboard")
# checker.save_all(OUTPUT_DIR, "checkerboard_40um_0deg")
save_fabrication_outputs(checker, "checkerboard_10um_0deg", arr)

C:\Users\qiaoz\AppData\Local\Temp\ipykernel_35780\3286779933.py:38: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


shape y,x: (5200, 5200) transmitting fraction: 0.5


{'gds': WindowsPath('generated_patterns/checkerboard_10um_0deg.gds'),
 'center_png': WindowsPath('generated_patterns/checkerboard_10um_0deg_center.png')}

## Focused X-ray Dammann grating

`DammannGrating` is a crossed binary-phase beam splitter for an odd square array such as 5 x 5, used with an external focusing optic. The Dammann period is fixed by X-ray energy, focal length, and the requested beamlet separation: `period = wavelength * focal_length / separation`.

The **simulation grid** and the **fabrication grid** are configured independently below. The simulation uses a compact pupil with roughly eight Dammann periods so the focal spot is resolved quickly. The fabrication aperture may be much larger, but it retains the same physical Dammann period and a separately chosen pixel pitch. Full-aperture GDS export is disabled by default and uses a single-period GDS array reference, so it does not allocate the full raster in RAM.

The detector result overlays the nominal beamlet centers after Fresnel propagation; at large downstream distances the defocused coherent beamlets can overlap. The fabrication raster is fixed to 100 nm before optimization: the Dammann period is snapped to an integer number of 100 nm pixels, and the same constrained pattern is used for simulation, tolerance analysis, and GDS export. At least 16 pixels per period are required. The tolerance sweep evaluates four realistic offsets from this constrained design: physical Dammann-period error, raster-pitch (quantization) error, phase-depth error, and actual focal-length error. The focal-length sweep reports the resulting beamlet-pitch error at the actual focal plane; it intentionally does not model detector-plane defocus.


In [6]:
# Beam-splitting design targets shared by both simulation and fabrication.
XRAY_ENERGY_KEV = 12.0
FOCAL_LENGTH_M = 0.4
DESIRED_BEAMLET_SEPARATION_UM = 20.0
ORDERS_PER_AXIS = 5

SAVE_FOCAL_ARRAYS = False  # Set True to retain the full padded intensity arrays.

file_prefix = f"dammann_{ORDERS_PER_AXIS}x{ORDERS_PER_AXIS}_f{int(FOCAL_LENGTH_M*1000)}mm_Sep{int(DESIRED_BEAMLET_SEPARATION_UM)}um"
print(f"File prefix: {file_prefix}")
suggested = DammannGrating.suggest_parameters(
    energy_keV=XRAY_ENERGY_KEV,
    focal_length_m=FOCAL_LENGTH_M,
    desired_separation_um=DESIRED_BEAMLET_SEPARATION_UM,
    orders_per_axis=ORDERS_PER_AXIS,
)

# Fabrication-constrained design grid. The 100 nm raster is used for both the Dammann optimization and its simulation.
FABRICATION_FIELD_SIZE_UM = (2000.0, 2000.0)
FABRICATION_RASTER_RESOLUTION_UM = 0.10  # 100 nm fixed fabrication and design raster.
FABRICATION_MIN_SAMPLES_PER_PERIOD = 16
FABRICATION_PERIOD_PIXELS = int(round(suggested["period_um"] / FABRICATION_RASTER_RESOLUTION_UM))
if FABRICATION_PERIOD_PIXELS < FABRICATION_MIN_SAMPLES_PER_PERIOD:
    raise ValueError("The requested beamlet spacing requires fewer than 16 pixels per Dammann period at 100 nm; increase the period or relax the spacing target.")
FABRICATION_EFFECTIVE_PERIOD_UM = FABRICATION_PERIOD_PIXELS * FABRICATION_RASTER_RESOLUTION_UM
FABRICATION_EFFECTIVE_SEPARATION_UM = suggested["wavelength_um"] * FOCAL_LENGTH_M * 1e6 / FABRICATION_EFFECTIVE_PERIOD_UM
FABRICATION_PERIOD_ERROR_PERCENT = 100.0 * (FABRICATION_EFFECTIVE_PERIOD_UM / suggested["period_um"] - 1.0)
SIMULATION_FIELD_PERIODS = 8
SIMULATION_FIELD_SIZE_UM = (SIMULATION_FIELD_PERIODS * FABRICATION_EFFECTIVE_PERIOD_UM,) * 2
SIMULATION_RASTER_RESOLUTION_UM = FABRICATION_RASTER_RESOLUTION_UM
SIMULATION_PUPIL_DIAMETER_UM = SIMULATION_FIELD_SIZE_UM[0]
SIMULATION_PADDING_FACTOR = max(4, int(suggested["focal_padding_factor"]))  # FFT zero-padding only refines focal-plane sampling.

# Detector is measured relative to the nominal focal plane.
DETECTOR_DISTANCE_AFTER_FOCUS_M = 1.0
DETECTOR_PADDING_FACTOR = 2
GENERATE_FABRICATION_GDS = True  # Set True only when the full design is ready to export.

for key in (
    "period_um",
    "recommended_simulation_raster_resolution_um",
    "recommended_simulation_field_size_um",
    "predicted_diffraction_limited_fwhm_um",
    "separation_to_fwhm_ratio",
):
    print(f"suggested {key}: {suggested[key]:.6g}")
print("fabrication field (um):", FABRICATION_FIELD_SIZE_UM)
print("fabrication raster (um):", FABRICATION_RASTER_RESOLUTION_UM)
print("fabrication/design raster (um):", FABRICATION_RASTER_RESOLUTION_UM)
print("fabrication pixels per period:", FABRICATION_PERIOD_PIXELS, "(minimum required:", FABRICATION_MIN_SAMPLES_PER_PERIOD, ")")
print("fabrication-constrained effective period (um):", FABRICATION_EFFECTIVE_PERIOD_UM)
print("fabrication-constrained beamlet separation (um):", FABRICATION_EFFECTIVE_SEPARATION_UM)
print("period quantization error (%):", FABRICATION_PERIOD_ERROR_PERCENT)
print(f"standalone binary-lens outer zone: {suggested['standalone_binary_lens_outermost_zone_um']:.6g} um")

dammann_5x5 = DammannGrating(
    field_size_um=SIMULATION_FIELD_SIZE_UM,
    raster_resolution_um=SIMULATION_RASTER_RESOLUTION_UM,
    xray_energy_keV=XRAY_ENERGY_KEV,
    orders_per_axis=ORDERS_PER_AXIS,
    period_um=FABRICATION_EFFECTIVE_PERIOD_UM,
    focal_length_m=FOCAL_LENGTH_M,
    transition_count=5,
    optimization_iterations=1200,
    transition_seed=23,
    raster_chunk_rows=128,
    include_binary_fresnel_lens=False,  # A separate optic sets the focal length.
    supersample=1,
)
arr = show_mask(dammann_5x5, "Compact 5 x 5 Dammann simulation pupil")
report = dammann_5x5.diffraction_report()
print("optimized half-period transitions:", np.round(report["transitions_half_period"], 6))
print("predicted target-array uniformity (min/mean):", round(report["array_uniformity_min_over_mean"], 3))
print("first-order focal spacing (um):", round(report["first_order_spacing_um_at_focus"], 4))

focal_result = dammann_5x5.simulate_focal_plane(
    arr,
    model="ideal_focusing_optic",
    pupil_diameter_um=SIMULATION_PUPIL_DIAMETER_UM,
    pupil_shape="circular",
    padding_factor=SIMULATION_PADDING_FACTOR,
)
detector_result = dammann_5x5.simulate_detector_plane(
    DETECTOR_DISTANCE_AFTER_FOCUS_M,
    arr,
    pupil_diameter_um=SIMULATION_PUPIL_DIAMETER_UM,
    pupil_shape="circular",
    padding_factor=DETECTOR_PADDING_FACTOR,
)
print(f"detector plane: {DETECTOR_DISTANCE_AFTER_FOCUS_M:.3g} m after focus")
print(f"nominal detector beamlet spacing: {detector_result['first_order_spacing_um']:.4g} um")
print(f"nominal defocused beamlet envelope diameter: {detector_result['nominal_beamlet_envelope_diameter_um']:.4g} um")

print(f"File prefix: {file_prefix}")


if SAVE_FOCAL_ARRAYS:
    np.save(OUTPUT_DIR / f"{file_prefix}_focal_intensity.npy", focal_result["intensity"])
    np.save(OUTPUT_DIR / f"{file_prefix}_reference_focal_intensity.npy", focal_result["reference_intensity"])
    np.save(OUTPUT_DIR / f"{file_prefix}_order_power.npy", focal_result["order_power"])
simulation_metrics = {
    "model": focal_result["model"],
    "simulation_field_size_um": list(SIMULATION_FIELD_SIZE_UM),
    "simulation_raster_resolution_um": SIMULATION_RASTER_RESOLUTION_UM,
    "simulation_pupil_diameter_um": SIMULATION_PUPIL_DIAMETER_UM,
    "fabrication_field_size_um": list(FABRICATION_FIELD_SIZE_UM),
    "fabrication_raster_resolution_um": FABRICATION_RASTER_RESOLUTION_UM,
    "fabrication_constraints": {
        "design_and_fabrication_raster_um": FABRICATION_RASTER_RESOLUTION_UM,
        "period_pixels": FABRICATION_PERIOD_PIXELS,
        "minimum_samples_per_period": FABRICATION_MIN_SAMPLES_PER_PERIOD,
        "effective_period_um": FABRICATION_EFFECTIVE_PERIOD_UM,
        "effective_beamlet_separation_um": FABRICATION_EFFECTIVE_SEPARATION_UM,
        "period_quantization_error_percent": FABRICATION_PERIOD_ERROR_PERCENT,
    },
    "captured_target_power": focal_result["captured_target_power"],
    "order_uniformity_min_over_mean": focal_result["order_uniformity_min_over_mean"],
    "integration_radius_um": focal_result["integration_radius_um"],
    "focal_pixel_um": focal_result["focal_pixel_um"],
    "reference_fwhm_estimate_um": focal_result["reference_fwhm_estimate_um"],
    "order_coordinates_um": focal_result["order_coordinates_um"].tolist(),
    "detector_plane": {
        "distance_after_focus_m": detector_result["distance_after_focus_m"],
        "first_order_spacing_um": detector_result["first_order_spacing_um"],
        "nominal_beamlet_envelope_diameter_um": detector_result["nominal_beamlet_envelope_diameter_um"],
        "detector_pixel_um": detector_result["detector_pixel_um"],
        "order_coordinates_um": detector_result["order_coordinates_um"].tolist(),
    },
}
(OUTPUT_DIR / f"{file_prefix}_focal_simulation.json").write_text(json.dumps(simulation_metrics, indent=2), encoding="utf-8")

spot_extent_um = np.max(np.abs(focal_result["order_coordinates_um"])) + 1.5 * report["first_order_spacing_um_at_focus"]
reference_extent_um = max(4.0 * focal_result["reference_fwhm_estimate_um"], 0.5 * DESIRED_BEAMLET_SEPARATION_UM)
detector_extent_um = np.max(np.abs(detector_result["order_coordinates_um"])) + 0.6 * detector_result["nominal_beamlet_envelope_diameter_um"]
x_keep = np.abs(focal_result["focal_x_um"]) <= spot_extent_um
y_keep = np.abs(focal_result["focal_y_um"]) <= spot_extent_um
reference_x_keep = np.abs(focal_result["focal_x_um"]) <= reference_extent_um
reference_y_keep = np.abs(focal_result["focal_y_um"]) <= reference_extent_um
detector_x_keep = np.abs(detector_result["detector_x_um"]) <= detector_extent_um
detector_y_keep = np.abs(detector_result["detector_y_um"]) <= detector_extent_um
focused_crop = focal_result["intensity"][np.ix_(y_keep, x_keep)]
reference_crop = focal_result["reference_intensity"][np.ix_(reference_y_keep, reference_x_keep)]
detector_crop = detector_result["intensity"][np.ix_(detector_y_keep, detector_x_keep)]
center_row = np.argmin(np.abs(focal_result["focal_y_um"]))
detector_center_row = np.argmin(np.abs(detector_result["detector_y_um"]))
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes[0, 0].imshow(np.log10(reference_crop + np.finfo(float).tiny), cmap="magma", origin="lower", extent=(-reference_extent_um, reference_extent_um, -reference_extent_um, reference_extent_um), interpolation="nearest")
axes[0, 0].set_title("Reference diffraction-limited focal spot")
axes[0, 0].set_xlabel("focal x (um)")
axes[0, 0].set_ylabel("focal y (um)")
axes[0, 1].imshow(np.log10(focused_crop + np.finfo(float).tiny), cmap="magma", origin="lower", extent=(-spot_extent_um, spot_extent_um, -spot_extent_um, spot_extent_um), interpolation="nearest")
for y0 in focal_result["order_coordinates_um"]:
    for x0 in focal_result["order_coordinates_um"]:
        axes[0, 1].plot(x0, y0, "c+", markersize=7, markeredgewidth=1.0)
axes[0, 1].set_title("Simulated 5 x 5 split focal spots")
axes[0, 1].set_xlabel("focal x (um)")
axes[0, 1].set_ylabel("focal y (um)")
axes[0, 2].imshow(np.log10(detector_crop + np.finfo(float).tiny), cmap="magma", origin="lower", extent=(-detector_extent_um, detector_extent_um, -detector_extent_um, detector_extent_um), interpolation="nearest")
for y0 in detector_result["order_coordinates_um"]:
    for x0 in detector_result["order_coordinates_um"]:
        axes[0, 2].plot(x0, y0, "c+", markersize=6, markeredgewidth=0.9)
axes[0, 2].set_title(f"Detector plane, {DETECTOR_DISTANCE_AFTER_FOCUS_M:.2f} m after focus")
axes[0, 2].set_xlabel("detector x (um)")
axes[0, 2].set_ylabel("detector y (um)")
axes[1, 0].plot(focal_result["focal_x_um"][x_keep], focal_result["reference_intensity"][center_row, x_keep] / focal_result["reference_intensity"][center_row].max(), label="reference spot")
axes[1, 0].plot(focal_result["focal_x_um"][x_keep], focal_result["intensity"][center_row, x_keep] / focal_result["intensity"][center_row].max(), label="Dammann split row")
axes[1, 0].set_title("Central focal-plane beam profile")
axes[1, 0].set_xlabel("focal x (um)")
axes[1, 0].set_ylabel("normalized intensity")
axes[1, 0].legend()
axes[1, 1].imshow(focal_result["order_power"], cmap="viridis", origin="lower")
axes[1, 1].set_title(f"Integrated focal order power, min/mean={focal_result['order_uniformity_min_over_mean']:.3f}")
axes[1, 1].set_xlabel("order index x")
axes[1, 1].set_ylabel("order index y")
axes[1, 1].set_xticks(range(dammann_5x5.orders_per_axis), focal_result["order_indices"])
axes[1, 1].set_yticks(range(dammann_5x5.orders_per_axis), focal_result["order_indices"])
axes[1, 2].plot(detector_result["detector_x_um"][detector_x_keep], detector_result["intensity"][detector_center_row, detector_x_keep] / detector_result["intensity"][detector_center_row].max())
for x0 in detector_result["order_coordinates_um"]:
    axes[1, 2].axvline(x0, color="c", alpha=0.45, linewidth=0.8)
axes[1, 2].set_title("Detector centerline; cyan = nominal beamlet centers")
axes[1, 2].set_xlabel("detector x (um)")
axes[1, 2].set_ylabel("normalized intensity")
fig.savefig(OUTPUT_DIR / f"{file_prefix}_focal_and_detector_simulation.png", dpi=220)
plt.show()
print("simulated captured target power:", round(focal_result["captured_target_power"], 3))
print("simulated order uniformity (min/mean):", round(focal_result["order_uniformity_min_over_mean"], 3))

if GENERATE_FABRICATION_GDS:
    fabrication_paths = save_periodic_dammann_gds(
        dammann_5x5,
        f"{file_prefix}_fabrication",
        FABRICATION_FIELD_SIZE_UM,
        FABRICATION_RASTER_RESOLUTION_UM,
        min_pixels_per_period=FABRICATION_MIN_SAMPLES_PER_PERIOD,
    )
    print("saved periodic full-aperture GDS:", fabrication_paths)
else:
    print("full-aperture GDS export is disabled; set GENERATE_FABRICATION_GDS=True when ready")


File prefix: dammann_5x5_f400mm_Sep20um
suggested period_um: 2.0664
suggested recommended_simulation_raster_resolution_um: 0.0161438
suggested recommended_simulation_field_size_um: 16.5312
suggested predicted_diffraction_limited_fwhm_um: 2.57
suggested separation_to_fwhm_ratio: 7.7821
fabrication field (um): (2000.0, 2000.0)
fabrication raster (um): 0.1
fabrication/design raster (um): 0.1
fabrication pixels per period: 21 (minimum required: 16 )
fabrication-constrained effective period (um): 2.1
fabrication-constrained beamlet separation (um): 19.680000000000003
period quantization error (%): 1.6260162601625883
standalone binary-lens outer zone: 2.5 um
shape y,x: (168, 168) transmitting fraction: 0.4444444444444444
optimized half-period transitions: [0.095578 0.121013 0.384052 0.484938 0.493056]
predicted target-array uniformity (min/mean): 0.892
first-order focal spacing (um): 19.68
detector plane: 1 m after focus
nominal detector beamlet spacing: 68.88 um
nominal defocused beamlet en

C:\Users\qiaoz\AppData\Local\Temp\ipykernel_37916\3724285857.py:40: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
C:\Users\qiaoz\AppData\Local\Temp\ipykernel_37916\288408112.py:184: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


simulated captured target power: 0.222
simulated order uniformity (min/mean): 0.708
saved periodic full-aperture GDS: {'gds': WindowsPath('generated_patterns/dammann_5x5_f400mm_Sep20um_fabrication.gds'), 'center_png': WindowsPath('generated_patterns/dammann_5x5_f400mm_Sep20um_fabrication_center.png'), 'metadata': WindowsPath('generated_patterns/dammann_5x5_f400mm_Sep20um_fabrication.json')}


### Dammann beam-splitting tolerance sweep

The four curves use the compact simulation pupil above, so they are suitable for rapid parameter studies. `period` changes the physical Dammann period, `raster` changes the fabrication pixel pitch while preserving the nominal period, and `phase depth` changes the phase step from its ideal pi value. `focal length` changes the focal-plane beamlet spacing expected from the external optic. The plots report target-order power, min/mean uniformity, and beamlet-pitch error relative to the requested separation.


In [4]:
# Relative departures from the nominal design. Add denser points here for a detailed tolerance curve.
TOLERANCE_FACTORS = np.array([0.90, 0.95, 1.00, 1.05, 1.10])
TOLERANCE_PADDING_FACTOR = 2
nominal_transitions = dammann_5x5.design_transitions()
tolerance_rows = []

for parameter_name in ("period", "raster", "phase_depth", "focal_length"):
    for factor in TOLERANCE_FACTORS:
        trial_period_um = dammann_5x5.period_um * (factor if parameter_name == "period" else 1.0)
        trial_raster_um = SIMULATION_RASTER_RESOLUTION_UM * (factor if parameter_name == "raster" else 1.0)
        trial_focal_length_m = FOCAL_LENGTH_M * (factor if parameter_name == "focal_length" else 1.0)
        trial_phase_depth_rad = np.pi * (factor if parameter_name == "phase_depth" else 1.0)
        trial = DammannGrating(
            field_size_um=SIMULATION_FIELD_SIZE_UM,
            raster_resolution_um=trial_raster_um,
            xray_energy_keV=XRAY_ENERGY_KEV,
            orders_per_axis=ORDERS_PER_AXIS,
            period_um=trial_period_um,
            focal_length_m=trial_focal_length_m,
            transition_count=len(nominal_transitions),
            optimization_iterations=0,
            transitions_half_period=nominal_transitions,
            raster_chunk_rows=128,
            include_binary_fresnel_lens=False,
            supersample=1,
        )
        trial_result = trial.simulate_focal_plane(
            model="ideal_focusing_optic",
            pupil_diameter_um=SIMULATION_PUPIL_DIAMETER_UM,
            pupil_shape="circular",
            padding_factor=TOLERANCE_PADDING_FACTOR,
            phase_depth_rad=trial_phase_depth_rad,
        )
        actual_pitch_um = float(trial_result["order_coordinates_um"][1] - trial_result["order_coordinates_um"][0])
        tolerance_rows.append({
            "parameter": parameter_name,
            "factor": float(factor),
            "percent_offset": float(100.0 * (factor - 1.0)),
            "period_um": float(trial_period_um),
            "raster_resolution_um": float(trial_raster_um),
            "phase_depth_rad": float(trial_phase_depth_rad),
            "focal_length_m": float(trial_focal_length_m),
            "captured_target_power": float(trial_result["captured_target_power"]),
            "order_uniformity_min_over_mean": float(trial_result["order_uniformity_min_over_mean"]),
            "actual_beamlet_pitch_um": actual_pitch_um,
            "beamlet_pitch_error_percent": float(100.0 * (actual_pitch_um / DESIRED_BEAMLET_SEPARATION_UM - 1.0)),
        })

(OUTPUT_DIR / "dammann_5x5_f300mm_tolerance_sweep.json").write_text(json.dumps(tolerance_rows, indent=2), encoding="utf-8")
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
labels = {
    "period": "period",
    "raster": "raster pitch",
    "phase_depth": "phase depth",
    "focal_length": "focal length",
}
for parameter_name, label in labels.items():
    rows = [row for row in tolerance_rows if row["parameter"] == parameter_name]
    offsets = [row["percent_offset"] for row in rows]
    axes[0].plot(offsets, [row["captured_target_power"] for row in rows], "o-", label=label)
    axes[1].plot(offsets, [row["order_uniformity_min_over_mean"] for row in rows], "o-", label=label)
    axes[2].plot(offsets, [row["beamlet_pitch_error_percent"] for row in rows], "o-", label=label)
axes[0].set_title("Captured target-order power")
axes[0].set_ylabel("fraction of incident power")
axes[1].set_title("Target-order uniformity")
axes[1].set_ylabel("minimum / mean")
axes[2].set_title("Beamlet-pitch error")
axes[2].set_ylabel("relative to requested pitch (%)")
for axis in axes:
    axis.axvline(0.0, color="0.5", linewidth=0.8)
    axis.set_xlabel("parameter offset (%)")
    axis.grid(alpha=0.25)
axes[0].legend(loc="best")
fig.savefig(OUTPUT_DIR / "dammann_5x5_f300mm_tolerance_sweep.png", dpi=220)
plt.show()
for parameter_name in labels:
    nominal = next(row for row in tolerance_rows if row["parameter"] == parameter_name and row["factor"] == 1.0)
    print(f"{labels[parameter_name]} nominal: power={nominal['captured_target_power']:.3f}, uniformity={nominal['order_uniformity_min_over_mean']:.3f}, pitch={nominal['actual_beamlet_pitch_um']:.3f} um")


period nominal: power=0.310, uniformity=0.679, pitch=5.000 um
raster pitch nominal: power=0.310, uniformity=0.679, pitch=5.000 um
phase depth nominal: power=0.310, uniformity=0.679, pitch=5.000 um
focal length nominal: power=0.310, uniformity=0.679, pitch=5.000 um


C:\Users\qiaoz\AppData\Local\Temp\ipykernel_24408\995506587.py:75: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 1D silicon kinoform CRL array

`KinoformCRL1D` is for X-rays incident **parallel to the material surface**. Its surface layout is a binary plan-view kinoform etched vertically to one fixed depth, not a multilevel depth-relief lens. The 2-pi material path thickness is `h_2pi = wavelength / delta`; for Si at 20 keV it is approximately 51.23 um when xraydb optical constants are available. `fixed_etch_depth_um` sets the common vertical etch depth.

Each cylindrical CRL focuses in x and the serial CRL train runs along y. Every CRL cell uses two mirrored concave kinoform surfaces, with a retained central silicon wall of `central_wall_um`; its physical y length is therefore `2 * surface_length + central_wall_um`. Each surface uses `x^2 / (2 R)`, so the pair gives the required `x^2 / R` thickness variation and `f = R / (2 * N * delta)`. `array_count` places complete trains side-by-side along x, separated by `array_spacing_um`, to make parallel kinoform arrays without changing the focal length. The mask is an **etch mask**: etched pixels form the empty volume around the retained concave kinoform material. `frame_margin_um` leaves an unetched outer silicon frame, and `support_web_width_um` leaves lateral rails around every aperture to retain the CRLs mechanically. The GDS uses layer 1 for the fixed-depth etch and can optionally place an outline annotation on layer 100.


In [6]:
# Parallel-incidence 1D kinoform CRL: binary surface mask and fixed vertical etch depth.
KINOFORM_MATERIAL = 'Si'
KINOFORM_DENSITY_G_CM3 = 2.329
# KINOFORM_MATERIAL = 'C'
# KINOFORM_DENSITY_G_CM3 = 3.52
KINOFORM_RADIUS_UM = 10.0
KINOFORM_ENERGY_KEV = 20.0
KINOFORM_NUMBER_OF_CRLS = 30
KINOFORM_CRL_SPACING_UM = 10.0
KINOFORM_ARRAY_COUNT = 1  # Complete serial CRL trains, aligned side-by-side along x.
KINOFORM_ARRAY_SPACING_UM = 50.0  # Clear gap between adjacent parallel arrays.
KINOFORM_APERTURE_UM = 250.0
KINOFORM_SUPPORT_WIDTH_UM = 300.0
KINOFORM_RASTER_UM = 0.10  # 100 nm fixed-depth lithography raster.
KINOFORM_FIXED_ETCH_DEPTH_UM = 150.0
KINOFORM_CENTRAL_WALL_UM = 15.0  # Retained Si between the two kinoform surfaces of every CRL.
KINOFORM_SUPPORT_FRAME_UM = 10.0  # Retained silicon border around the entire kinoform layout.
KINOFORM_SUPPORT_WEB_UM = 2.0  # Retained silicon rail at each aperture edge.
KINOFORM_GDS_OUTLINE_WIDTH_UM = 0.0  # Set positive only to add a layer-100 annotation outline.

file_prefix = f"{KINOFORM_MATERIAL}_kinoform_1d_crl_parallel_r{int(KINOFORM_RADIUS_UM)}um_e{int(KINOFORM_ENERGY_KEV)}keV_n{KINOFORM_NUMBER_OF_CRLS}_s{int(KINOFORM_CRL_SPACING_UM)}um_a{KINOFORM_ARRAY_COUNT}_Aper{int(KINOFORM_APERTURE_UM)}um"

kinoform_crl = KinoformCRL1D(radius_um=KINOFORM_RADIUS_UM, xray_energy_keV=KINOFORM_ENERGY_KEV, number_of_crls=KINOFORM_NUMBER_OF_CRLS, crl_spacing_um=KINOFORM_CRL_SPACING_UM, array_count=KINOFORM_ARRAY_COUNT, array_spacing_um=KINOFORM_ARRAY_SPACING_UM, aperture_um=KINOFORM_APERTURE_UM, support_width_um=KINOFORM_SUPPORT_WIDTH_UM, raster_resolution_um=KINOFORM_RASTER_UM, fixed_etch_depth_um=KINOFORM_FIXED_ETCH_DEPTH_UM, central_wall_um=KINOFORM_CENTRAL_WALL_UM, frame_margin_um=KINOFORM_SUPPORT_FRAME_UM, support_web_width_um=KINOFORM_SUPPORT_WEB_UM, material=KINOFORM_MATERIAL, density_g_cm3=KINOFORM_DENSITY_G_CM3)
kinoform_surface_mask = kinoform_crl.surface_mask()
kinoform_etch_depth_um = kinoform_crl.etch_depth_map_um()
kinoform_profile_um = kinoform_crl.surface_profile_um()
kinoform_metadata = kinoform_crl.metadata()
kinoform_stem = file_prefix

np.save(OUTPUT_DIR / f'{kinoform_stem}_surface_mask.npy', kinoform_surface_mask)
np.save(OUTPUT_DIR / f'{kinoform_stem}_fixed_etch_depth_um.npy', kinoform_etch_depth_um)
(OUTPUT_DIR / f'{kinoform_stem}.json').write_text(json.dumps(kinoform_metadata, indent=2), encoding='utf-8')
save_fixed_depth_gds(kinoform_surface_mask, OUTPUT_DIR / f'{kinoform_stem}.gds', KINOFORM_RASTER_UM, kinoform_stem, frame_size_um=kinoform_crl.frame_rectangle_um(), frame_width_um=KINOFORM_GDS_OUTLINE_WIDTH_UM, frame_layer=100)

height_um, width_um = kinoform_crl.field_size_um()
fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
axes[0].imshow(kinoform_surface_mask, cmap='gray', vmin=0, vmax=1, origin='lower', interpolation='nearest', extent=(-width_um / 2, width_um / 2, -height_um / 2, height_um / 2), aspect='auto')
axes[0].plot([-width_um / 2, width_um / 2, width_um / 2, -width_um / 2, -width_um / 2], [-height_um / 2, -height_um / 2, height_um / 2, height_um / 2, -height_um / 2], color='tab:red', linewidth=0.8)
axes[0].set(title='Fixed-depth etch mask (white = etched; black = retained Si)', xlabel='focusing x (um)', ylabel='CRL-train y (um)')
x_profile = (np.arange(kinoform_surface_mask.shape[1]) + 0.5) * KINOFORM_RASTER_UM - width_um / 2
axes[1].plot(x_profile, kinoform_profile_um)
axes[1].set(title='Single-surface kinoform path profile', xlabel='focusing x (um)', ylabel='surface path length (um)')
axes[1].grid(alpha=0.25)
fig.savefig(OUTPUT_DIR / f'{kinoform_stem}_layout.png', dpi=600)
plt.show()

print(f'material: {KINOFORM_MATERIAL}; incidence: parallel to surface')
print('delta:', f"{kinoform_metadata['delta']:.6g}")
print('2pi material path thickness lambda/delta (um):', f"{kinoform_metadata['two_pi_path_jump_um']:.6g}")
print('fixed vertical etch depth (um):', f"{kinoform_metadata['fixed_etch_depth_um']:.6g}")
print('focal length f = R / (2 * N * delta):', f"{kinoform_metadata['focal_length_m']:.6g} m")
print('saved:', OUTPUT_DIR / f'{kinoform_stem}.gds')


material: Si; incidence: parallel to surface
delta: 1.21006e-06
2pi material path thickness lambda/delta (um): 51.2303
fixed vertical etch depth (um): 150
focal length f = R / (2 * N * delta): 0.137734 m
saved: generated_patterns\Si_kinoform_1d_crl_parallel_r10um_e20keV_n30_s10um_a1_Aper250um.gds


C:\Users\qiaoz\AppData\Local\Temp\ipykernel_26864\4088588671.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Kinoform focal-spot resolution and coherence simulation

This one-dimensional scalar simulation compares the ideal continuous kinoform phase with the fixed-depth surface pattern sampled at a finite lithography resolution. Illumination is a Gaussian beam. The fully coherent result uses a point source; the partially coherent result incoherently averages Gaussian-distributed source-angle modes for a 30 um FWHM source at 30 m. The 100 nm fabrication raster changes only the in-plane pattern sampling, not the fixed vertical etch depth.


In [4]:
KINOFORM_FABRICATION_RESOLUTION_UM = 0.20
KINOFORM_SIMULATION_RESOLUTION_UM = 0.2
GAUSSIAN_INTENSITY_FWHM_UM = KINOFORM_APERTURE_UM
SOURCE_SIZE_FWHM_UM = 30.0
SOURCE_DISTANCE_M = 30.0
PARTIAL_COHERENCE_MODES = 15

kinoform_focus = kinoform_crl.simulate_focal_spot(fabrication_resolution_um=KINOFORM_FABRICATION_RESOLUTION_UM, simulation_resolution_um=KINOFORM_SIMULATION_RESOLUTION_UM, gaussian_intensity_fwhm_um=GAUSSIAN_INTENSITY_FWHM_UM, source_size_fwhm_um=SOURCE_SIZE_FWHM_UM, source_distance_m=SOURCE_DISTANCE_M, source_modes=PARTIAL_COHERENCE_MODES, padding_factor=4)
(OUTPUT_DIR / f'{kinoform_stem}_focal_simulation.json').write_text(json.dumps({key: value for key, value in kinoform_focus.items() if key not in {'x_um', 'focal_x_um', 'ideal_profile_um', 'fabricated_profile_um', 'intensities'}}, indent=2), encoding='utf-8')

x_plot = kinoform_focus['x_um']
focal_x = kinoform_focus['focal_x_um']
focus_window_um = max(4.0, 8.0 * max(kinoform_focus['fwhm_um'].values()))
focal_keep = np.abs(focal_x) <= focus_window_um
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(x_plot, kinoform_focus['ideal_profile_um'], label='ideal continuous')
axes[0, 0].step(x_plot, kinoform_focus['fabricated_profile_um'], where='mid', label=f'{KINOFORM_FABRICATION_RESOLUTION_UM * 1000:.0f} nm sampled')
axes[0, 0].set(title='In-plane kinoform profile', xlabel='focusing x (um)', ylabel='path profile (um)')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.25)
for key, label, color in [('ideal_coherent', 'ideal', 'black'), ('fabricated_coherent', '100 nm fabricated', 'tab:blue')]:
    axes[0, 1].plot(focal_x[focal_keep], kinoform_focus['intensities'][key][focal_keep] / kinoform_focus['intensities'][key].max(), label=label, color=color)
axes[0, 1].set(title='Fully coherent focal spot', xlabel='focal x (um)', ylabel='normalized intensity')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.25)
for key, label, color in [('ideal_partial', 'ideal', 'black'), ('fabricated_partial', '100 nm fabricated', 'tab:orange')]:
    axes[1, 0].plot(focal_x[focal_keep], kinoform_focus['intensities'][key][focal_keep] / kinoform_focus['intensities'][key].max(), label=label, color=color)
axes[1, 0].set(title=f'Partial coherence: {SOURCE_SIZE_FWHM_UM:.0f} um source at {SOURCE_DISTANCE_M:.0f} m', xlabel='focal x (um)', ylabel='normalized intensity')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.25)
metric_names = ['ideal_coherent', 'fabricated_coherent', 'ideal_partial', 'fabricated_partial']
metric_labels = ['ideal\ncoherent', '100 nm\ncoherent', 'ideal\npartial', '100 nm\npartial']
axes[1, 1].bar(metric_labels, [kinoform_focus['fwhm_um'][name] for name in metric_names], color=['black', 'tab:blue', '0.35', 'tab:orange'])
axes[1, 1].set(title='Focal-spot FWHM', ylabel='FWHM (um)')
axes[1, 1].grid(axis='y', alpha=0.25)
fig.savefig(OUTPUT_DIR / f'{kinoform_stem}_ideal_vs_100nm_focal_spots.png', dpi=220)
plt.show()

print('focal length (m):', f"{kinoform_focus['focal_length_m']:.6g}")
print('partial-coherence source blur sigma at focus (um):', f"{kinoform_focus['source_blur_sigma_um']:.4g}")
print('100 nm coherent relative peak:', f"{kinoform_focus['relative_peak']['fabricated_over_ideal_coherent']:.4f}")
print('100 nm partial-coherence relative peak:', f"{kinoform_focus['relative_peak']['fabricated_over_ideal_partial']:.4f}")
print('FWHM (um):', {name: round(value, 4) for name, value in kinoform_focus['fwhm_um'].items()})


focal length (m): 0.302519
partial-coherence source blur sigma at focus (um): 0.1285
100 nm coherent relative peak: 1.0000
100 nm partial-coherence relative peak: 1.0000
FWHM (um): {'ideal_coherent': 0.1786, 'fabricated_coherent': 0.1786, 'ideal_partial': 0.3572, 'fabricated_partial': 0.3572}


C:\Users\qiaoz\AppData\Local\Temp\ipykernel_40392\625299566.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Planned carrier-grating library

This cell gives the notebook direct access to every carrier-family realization in `plan.md`: S4, H3, H3-4L, DRH6, FATO and MPG proxies, square/hexagonal CUC, RBN, grid, checkerboard, and the two-shot control. It writes each small periodic phase map and propagated intensity to `generated_patterns/planned_grating_library`. These are simulation cells; use `export_fabrication_patterns.py` for the optimized full-aperture S4/H3/CUC layouts, and the Dammann cell above for its focused GDS layout.

In [ ]:
library_config = DesignConfig(
    supercell_um=56.0,
    samples_per_supercell=112,
    n_supercells=2,
    wavelength_m=1.2398419843320026e-9 / 14.0,
    z_m=0.4,
    phase_depth_rad=np.pi,
)
library_dir = OUTPUT_DIR / "planned_grating_library"
library_dir.mkdir(exist_ok=True)
simulations = [simulate_pattern(library_config, spec) for spec in default_patterns(library_config)]
for simulation in simulations:
    name = simulation["pattern"]["name"]
    np.save(library_dir / f"{name}_phase_map_rad.npy", simulation["phase_map"])
    np.save(library_dir / f"{name}_intensity.npy", simulation["intensity"])

fig, axes = plt.subplots(3, 4, figsize=(12, 9), constrained_layout=True)
for ax, simulation in zip(axes.flat, simulations):
    ax.imshow(simulation["phase_map"], cmap="twilight", origin="lower", interpolation="nearest")
    ax.set_title(simulation["pattern"]["name"])
    ax.set_xticks([])
    ax.set_yticks([])
fig.savefig(library_dir / "planned_grating_phase_library.png", dpi=180)
plt.show()
print(f"Saved {len(simulations)} planned grating realizations to {library_dir}")

## Regular Hartmann plate

Important parameters:

- `period_um`: center-to-center lattice pitch.
- `lattice`: `"square"` or `"hexagonal"`.
- `hole_shape`: `"circle"`, `"square"`, or `"regular_polygon"`.
- `hole_size_um`: diameter for circles or width for squares. If omitted, it is `period_um * duty_cycle`.
- `duty_cycle`: hole size divided by period when `hole_size_um` is not provided.
- `hole_rotation_deg`: rotates each square/polygon hole in degrees. Rotation is invisible for circular holes.
- `hole_rotation`: optional alias for `hole_rotation_deg`, useful when building parameter dictionaries.
- `array_rotation_deg`: rotates the whole lattice.

In [8]:
hartmann = HartmannPlate(
    field_size_um=(1000, 1000),
    raster_resolution_um=0.2,
    xray_energy_keV=12.0,
    period_um=60.0,
    lattice="square",
    hole_shape="square",
    hole_size_um=28.0,
    hole_rotation_deg=25.0,
    supersample=1,
)
arr = show_mask(hartmann, "square Hartmann plate, 25 degree hole rotation")
# hartmann.save_all(OUTPUT_DIR, "hartmann_square_120um_pitch_hole_rot25")
save_fabrication_outputs(hartmann, "hartmann_square_60um_pitch_hole_rot25_1mmx1mm", arr)

C:\Users\qiaoz\AppData\Local\Temp\ipykernel_84304\3724285857.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


shape y,x: (5000, 5000) transmitting fraction: 0.21340044


{'gds': WindowsPath('generated_patterns/hartmann_square_60um_pitch_hole_rot25_1mmx1mm.gds'),
 'center_png': WindowsPath('generated_patterns/hartmann_square_60um_pitch_hole_rot25_1mmx1mm_center.png')}

## Random binary speckle mask

Important parameters:

- `pitch_um`: size of one independent random square cell. Match this to the desired speckle feature size.
- `fill_fraction`: probability that a cell transmits.
- `seed`: fixed seed for reproducibility. Change it to get another random realization.
- `connected`: when `True`, attempts to connect opaque material islands in the low-resolution cell map. This is useful for free-standing masks, but changes the exact fill fraction.
- `supersample`: use `1` here because random cells are axis-aligned squares and should remain exact.

In [24]:
speckle = RandomBinaryMask(
    field_size_um=(13000, 13000),
    raster_resolution_um=5,
    xray_energy_keV=14.0,
    pitch_um=5.0,
    fill_fraction=0.5,
    seed=42,
    connected=False,
    supersample=1,
)
arr = show_mask(speckle, "random binary speckle mask")
# speckle.save_all(OUTPUT_DIR, "random_binary_pitch5um_seed42")
save_fabrication_outputs(speckle, "random_binary_pitch5um_seed42", arr)

C:\Users\qiaoz\AppData\Local\Temp\ipykernel_35780\3286779933.py:38: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


shape y,x: (2600, 2600) transmitting fraction: 0.5002156804733727


{'gds': WindowsPath('generated_patterns/random_binary_pitch5um_seed42.gds'),
 'center_png': WindowsPath('generated_patterns/random_binary_pitch5um_seed42_center.png')}

## Random Hartmann plate

Important parameters:

- `n_holes`: requested hole count. Use this or `target_density_per_mm2`.
- `target_density_per_mm2`: density alternative to `n_holes`.
- `min_center_distance_um`: minimum center spacing. The generator also checks actual footprint overlap using bounding radius.
- `edge_clearance_um`: minimum gap from hole footprint to field edge.
- `hole_size_um`: scalar size or distribution, for example `{"type": "uniform", "low": 15, "high": 25}`.
- `hole_rotation`: fixed angle, `"random"`, or distribution dict.
- `n_cores`: number of worker processes for a full `supersample=1` raster. Use the sparse preview and vector-GDS export below for large fabrication fields; those paths do not materialize the full raster.
- `seed`: controls the explicit hole layout.

The metadata JSON stores the final `holes` list with `x_um`, `y_um`, `hole_size_um`, `rotation_deg`, and `bounding_radius_um`.

For a minimum center distance `d`, the absolute hexagonal-packing density limit is `2e6 / (sqrt(3) * d^2)` holes/mm2. At `d=113 um`, this is only 90.5 holes/mm2, so 160 holes/mm2 cannot be placed. A random sequential layout must remain well below this geometric bound; the cell below uses 45 holes/mm2 for reliable placement.

In [ ]:
random_h = RandomHartmann(
    field_size_um=(13000, 13000),
    raster_resolution_um=4,
    xray_energy_keV=14.0,
    n_holes=2000,
    hole_rotation=25.0,
    min_center_distance_um=70.0,
    hole_shape="square",
    hole_size_um={"type": "uniform", "low": 35.0, "high": 35.0},
    edge_clearance_um=20.0,
    seed=7,
    supersample=1,
)
arr = show_mask(random_h, "random Hartmann plate")
# random_h.save_all(OUTPUT_DIR, "random_hartmann_250holes_seed7")
save_random_hartmann_outputs(random_h, "random_hartmann_250holes", arr)

## Analytic Random Hartmann first guess

Use this helper before choosing Random Hartmann parameters. It estimates hole size and spacing from energy, distance, source blur, detector pixel size, and desired spot separation.

Inputs:

- `energy_keV`: X-ray energy.
- `mask_to_detector_m`: mask-to-detector distance `L2`.
- `pixel_size_um`: detector pixel pitch.
- `source_to_mask_m`: source-to-mask distance `L1`, or `"inf"` for collimated.
- `source_size_um`: source FWHM. Ignored for collimated beams.
- `detector_psf_fwhm_um`: detector point-spread FWHM.
- `min_feature_um`: fabrication lower bound.
- `max_slope_urad`: maximum expected wavefront slope; larger values force more separation.
- `separation_factor`: multiplier on spot FWHM for non-overlap. Use at least `3`.

In [2]:
guess = analytic_hartmann_first_guess(
    energy_keV=8,
    mask_to_detector_m=1.5,
    pixel_size_um=6.5,
    source_to_mask_m="inf",
    source_size_um=0.0,
    detector_psf_fwhm_um=0.0,
    min_feature_um=10.0,
    max_slope_urad=10.0,
    separation_factor=3.0,
)
guess

{'wavelength_nm': 0.15498,
 'magnification': 1.0,
 'hole_size_um': 23.816523675801218,
 'spot_fwhm_um': 33.681650790898004,
 'spot_fwhm_pixels': 5.181792429368924,
 'min_center_distance_um': 131.044952372694,
 'detector_min_spacing_um': 131.044952372694}

In [4]:
optimized_seed_mask = RandomHartmann(
    field_size_um=(13000, 13000),
    raster_resolution_um=0.2,
    xray_energy_keV=0.50,
    target_density_per_mm2=260,  # 160 is infeasible with 113 um minimum spacing; theoretical limit is 90.5/mm2.
    hole_shape="square",
    hole_rotation=25.0,
    min_center_distance_um=50,
    hole_size_um=20,
    edge_clearance_um=20,
    seed=123,
    supersample=1,
    n_cores=max(1, (os.cpu_count() or 2) - 1),
    chunk_rows=256,
)
limits = optimized_seed_mask.placement_limits()
print('Random Hartmann placement reference')
print('  requested density / hole count:', f"{limits['requested_density_per_mm2']:.2f} holes/mm2 / {limits['requested_hole_count']}")
print('  effective min center distance:', f"{limits['effective_min_center_distance_um']:.2f} um", '(includes the rotated-square footprint)')
print('  theoretical hexagonal upper density:', f"{limits['theoretical_hexagonal_max_density_per_mm2']:.2f} holes/mm2")
print('  recommended random-sequential upper:', f"{limits['random_sequential_reference_density_per_mm2']:.2f} holes/mm2 / {limits['random_sequential_reference_hole_count']} holes")
print('  full 0.2 um raster:', limits['full_raster_shape_yx'], f"({limits['full_boolean_raster_gib']:.2f} GiB bool); sparse GDS export does not create it")
if limits['requested_density_per_mm2'] > limits['random_sequential_reference_density_per_mm2']:
    print('  WARNING: requested density is above the random-sequential reference. Use target_density_per_mm2=110 for this generator, or use a structured/blue-noise layout for 130 holes/mm2.')
preview = show_mask(optimized_seed_mask, "analytic first-guess Random Hartmann")
# optimized_seed_mask.save_all(OUTPUT_DIR, "random_hartmann_analytic_guess")
save_random_hartmann_outputs(optimized_seed_mask, "random_hartmann_20umholes_50umMinDist_13mmsize", preview)

Random Hartmann placement reference
  requested density / hole count: 260.00 holes/mm2 / 43940
  effective min center distance: 50.00 um (includes the rotated-square footprint)
  theoretical hexagonal upper density: 461.88 holes/mm2
  recommended random-sequential upper: 267.89 holes/mm2 / 45273 holes
  full 0.2 um raster: [65000, 65000] (3.93 GiB bool); sparse GDS export does not create it


C:\Users\qiaoz\AppData\Local\Temp\ipykernel_26864\3271214691.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


preview shape y,x: (1200, 1200) fabrication shape y,x: (65000, 65000) preview transmitting fraction: 0.10389097222222222


{'gds': WindowsPath('generated_patterns/random_hartmann_20umholes_50umMinDist_13mmsize.gds'),
 'preview_png': WindowsPath('generated_patterns/random_hartmann_20umholes_50umMinDist_13mmsize_preview.png'),
 'metadata': WindowsPath('generated_patterns/random_hartmann_20umholes_50umMinDist_13mmsize.json')}

## Absorber and Talbot feasibility estimate

`absorber_requirements` returns approximate absorber thickness, phase thickness, aspect ratio, Talbot distances, and warnings. The fallback optical constants are for screening only; verify final mask fabrication against material data or xraylib.

In [17]:
absorber_requirements(
    energy_keV=14,
    material="Si",
    transmission_target=0.03,
    min_feature_um=5.0,
    grating_period_um=4.0,
    talbot_orders=(1, 2, 3, 4),
)

{'energy_keV': 14,
 'wavelength_nm': 0.08856,
 'material': 'Si',
 'beta_estimate': 6.1224489795918365e-09,
 'delta_estimate': 7.653061224489796e-07,
 'amplitude_thickness_um': 4036.301349826189,
 'pi_phase_thickness_um': 57.8592,
 'half_pi_phase_thickness_um': 28.9296,
 'warnings': ['Hard X-ray absorber aspect ratio exceeds 10:1.'],
 'amplitude_aspect_ratio': 807.2602699652377,
 'fractional_talbot_distances_m': {1: 0.02258355916892502,
  2: 0.04516711833785004,
  3: 0.06775067750677505,
  4: 0.09033423667570008}}

## Practical workflow

1. Start with `analytic_hartmann_first_guess` for Hartmann masks, or choose period/pitch directly for gratings and speckle masks.
2. Generate a small field first, inspect transmitting fraction and visual structure.
3. Increase `field_size_um` to the fabrication size only after the small test looks right.
4. Keep `seed` fixed for any mask you may fabricate.
5. Save with `save_all` and `save_fabrication_outputs`; the `.json` metadata is the reproducibility record and the `.gds` file is the compact fabrication layout.
6. For very large masks, raise `raster_resolution_um` for previews and only use fine resolution for final simulation crops.